# Read a LI-6400 / LI-6400XT over RS-232

Live-read measurement data from a LI-COR **LI-6400 / LI-6400XT** console over its **external RS-232 comm port**, through an RS-232-to-USB bridge.

> **Framing must match the console.** On the LI-6400: **Utility Menu → Configure the COMM port** shows the baud rate, data bits, parity, and handshaking. Put the same values in the *Configure* cell below.

## Two ways to get data out (set up on the console, then read here)

**A. Stream the log to the COMM port** — *recommended; gives the standard log record.*
Press **Log**; at the filename prompt press **Escape**; choose to log to the **COMM port** (answer the *“Log to COMM port?”* prompt / press **C**). For unattended fixed-interval runs use **AutoLog** / **Buffered AutoLog**. Every logged record then streams out as an ASCII line that this notebook captures (Step 5A).

**B. On-demand queries with LPL** — *advanced.*
From OPEN's main screen press **L** to enter **LTerm** mode (and/or enable *Execute as LPL* in the COMM config). Lines you send (newline-terminated) are compiled and executed as LPL. The exact command to print a value depends on your OPEN version (manual **Chapter 25**) — use `li.talk(...)` in Step 5B to experiment.

### Notebook steps
1. Discover the serial port
2. Configure the connection (match the console)
3. Confirm the link & auto-detect baud
4. Open a persistent connection
5. Read data — **A** (stream) or **B** (LPL query)
6. Clean up

In [6]:
import sys, time, datetime, csv

try:
    import serial
    import serial.tools.list_ports
except ImportError:
    raise SystemExit("pyserial is required.  Install it with:  pip install pyserial")

print("Python  :", sys.version.split()[0])
print("pyserial:", serial.__version__)

Python  : 3.13.5
pyserial: 3.5


## Step 1 — Discover the serial port
Run this and note the `COMx` name of your RS-232-to-USB bridge (often *USB Serial Port*, *CH340*, *CP210x*, *FT232*, *Prolific*, …).

In [7]:
ports = list(serial.tools.list_ports.comports())
if not ports:
    print("No serial ports found. Is the USB bridge plugged in and its driver installed?")
for p in ports:
    print(f"{p.device:<8}  {p.description}   [{p.hwid}]")

COM4      USB Serial Port (COM4)   [USB VID:PID=0403:6001 SER=B0048EB4A]


## Step 2 — Configure the connection
Set `PORT` to your bridge and the framing to match the console's COMM-port settings. If you don't know the baud rate yet, leave the default and use Step 3 to find it.

In [8]:
# --- Set these to match your bridge and the console's COMM-port settings ---
# Confirmed for this instrument: port COM4, and the console's Comm Port
# Configuration screen reports  "57600 8 1 N"  -> 57600 baud, 8 data bits,
# 1 stop bit, No parity.  (At 19200 you only get garbage.)

PORT      = "COM4"        # your RS-232-USB bridge (from Step 1)
BAUD      = 57600         # CONFIRMED from the console's Comm Port screen
DATA_BITS = 8             # 7 or 8
PARITY    = "N"           # "N", "E", or "O"
STOP_BITS = 1             # 1 or 2
XONXOFF   = False         # True if the console uses XON/XOFF (software) handshaking
RTSCTS    = False         # True if the console uses hardware handshaking

## Step 3 — Confirm the link & auto-detect baud
`scan_baud` listens at each common baud rate and reports which one yields clean text.

**Important:** the console must be *transmitting* during the scan. Set up streaming (Step 5A) or LTerm echo (Step 5B) first, otherwise every rate shows silence. Once you know the baud, put it in the *Configure* cell above.

In [9]:
_PRINTABLE = set(range(32, 127)) | {9, 10, 13}   # printable ASCII + tab/newline/CR

def _printable_ratio(data: bytes) -> float:
    if not data:
        return 0.0
    return sum(b in _PRINTABLE for b in data) / len(data)

def _framing(bits, parity, stop):
    return (
        {7: serial.SEVENBITS, 8: serial.EIGHTBITS}[bits],
        {"N": serial.PARITY_NONE, "E": serial.PARITY_EVEN, "O": serial.PARITY_ODD}[parity],
        {1: serial.STOPBITS_ONE, 2: serial.STOPBITS_TWO}[stop],
    )

def sniff(port, baud, seconds=2.5, framing=(8, "N", 1), xonxoff=False, rtscts=False):
    # Passively listen for `seconds`; return whatever raw bytes arrive.
    bits, par, stop = _framing(*framing)
    buf = bytearray()
    with serial.Serial(port, baud, bytesize=bits, parity=par, stopbits=stop,
                       xonxoff=xonxoff, rtscts=rtscts, timeout=0.2) as s:
        end = time.monotonic() + seconds
        while time.monotonic() < end:
            chunk = s.read(s.in_waiting or 1)
            if chunk:
                buf.extend(chunk)
    return bytes(buf)

def scan_baud(port, candidates=(9600, 19200, 38400, 57600, 115200, 4800, 2400),
              seconds=2.5, framing=(8, "N", 1)):
    # Try each baud and score how 'text-like' the received bytes look.
    # The console must be TRANSMITTING during the scan or every rate is silent.
    best = None
    print(f"{'baud':>7}  {'bytes':>6}  {'text':>5}   sample")
    for b in candidates:
        data = sniff(port, b, seconds, framing)
        ratio = _printable_ratio(data)
        sample = data[:40].decode("latin-1", "replace").replace("\r", " ").replace("\n", " ")
        print(f"{b:>7}  {len(data):>6}  {ratio:>4.0%}   {sample!r}")
        score = ratio if len(data) > 4 else 0.0
        if best is None or score > best[1]:
            best = (b, score, len(data))
    if best and best[2] > 4 and best[1] > 0.8:
        print(f"\nLikely baud: {best[0]}  (clean text, {best[2]} bytes)")
    else:
        print("\nNo clean text seen. Start a stream on the console (Step 5A), then "
              "re-run — or check framing / handshake / null-modem (see Troubleshooting).")
    return best[0] if best else None

In [10]:
# Start a stream on the console first (Step 5A) so there is traffic to catch.
# scan_baud(PORT)

## Step 4 — Open a persistent connection
`Li6400` keeps one serial connection open and gives you `watch()`, `talk()`, and `log_to_csv()`.

In [11]:
class Li6400:
    # Minimal RS-232 handle to a LI-6400/XT running OPEN.
    #   li = Li6400(PORT, BAUD)          open the link
    #   li.watch(5)                      print whatever streams for 5 s (liveness)
    #   li.talk("...", wait=1.0)         send an LPL line, return the reply (LTerm mode)
    #   li.log_to_csv("run.csv", 10)     capture the streamed log for 10 minutes
    #   li.close()

    def __init__(self, port, baud=19200, *, data_bits=8, parity="N", stop_bits=1,
                 xonxoff=False, rtscts=False, timeout=0.3):
        bits, par, stop = _framing(data_bits, parity, stop_bits)
        self.ser = serial.Serial(port, baud, bytesize=bits, parity=par, stopbits=stop,
                                 xonxoff=xonxoff, rtscts=rtscts,
                                 timeout=timeout, write_timeout=2.0)
        self.port, self.baud = port, baud

    def read_available(self, seconds=1.0) -> bytes:
        # Accumulate raw bytes for `seconds` (reassembles chunked USB reads).
        end = time.monotonic() + seconds
        buf = bytearray()
        while time.monotonic() < end:
            chunk = self.ser.read(self.ser.in_waiting or 1)
            if chunk:
                buf.extend(chunk)
        return bytes(buf)

    def watch(self, seconds=5.0) -> str:
        # Print raw text arriving over the next `seconds` (quick liveness check).
        text = self.read_available(seconds).decode("latin-1", "replace")
        print(text if text else "(silence — nothing received)")
        return text

    def send(self, line, eol="\n") -> None:
        # LPL executes on newline (decimal 10). Some setups want CR ('\r') or CRLF.
        self.ser.write((line + eol).encode())

    def talk(self, line, wait=1.0, eol="\n") -> str:
        # Send a line and return whatever comes back within `wait` seconds.
        self.ser.reset_input_buffer()
        self.send(line, eol)
        return self.read_available(wait).decode("latin-1", "replace")

    def log_to_csv(self, path, minutes=5.0, echo=True) -> None:
        # Append '<iso-timestamp>,<raw line>' for each newline-terminated record the
        # console streams. Use with the 'log to COMM port' route (Step 5A).
        end = time.monotonic() + minutes * 60
        pending = ""
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            print(f"logging to {path} for {minutes:g} min ... (interrupt the cell to stop)")
            while time.monotonic() < end:
                data = self.read_available(1.0)
                if not data:
                    continue
                text = pending + data.decode("latin-1", "replace")
                *lines, pending = text.split("\n")      # hold back any partial last line
                for ln in lines:
                    ln = ln.rstrip("\r")
                    if not ln.strip():
                        continue
                    ts = datetime.datetime.now().isoformat(timespec="milliseconds")
                    w.writerow([ts, ln])
                    f.flush()
                    if echo:
                        print(ts, ln)

    def close(self) -> None:
        try:
            self.ser.close()
        except Exception:
            pass

In [12]:
li = Li6400(PORT, BAUD , data_bits=DATA_BITS, parity=PARITY, stop_bits=STOP_BITS,
            xonxoff=XONXOFF, rtscts=RTSCTS)
print(f"opened {li.port} @ {li.baud} baud  {DATA_BITS}{PARITY}{STOP_BITS}")

# Immediate liveness check: is anything arriving right now?
li.watch(3)

opened COM4 @ 57600 baud  8N1
 Ã?à@    48.02 Û   PhotoÞÛ    CondÞÛ      CiÞÛ  TrmmolÞc  -0.215  -0.000165  -1.64E+03  -0.0026 LeafFan  Flow    Mixer   Temp    Lamp  2  1V     OFF      OFF     OFF   -none- A
@                                                                                                                                  R
   T
@ÛCO2R_æmlÞÛCO2S_æmlÞÛH2OR_mmlÞÛH2OS_mmlÞa   497.3     434.4    16.142    16.890      >> The Pump Is OFF... <<           b   -62.8     0.747      -2.1     47.99 Û   PhotoÞÛ    CondÞÛ      CiÞÛ  TrmmolÞc  -0.214  -0.00016  -1.69E+03  -0.00261 LeafFan  Flow    Mixer   Temp    Lamp  2  1V     OFF      OFF     OFF   -none- A
@                                               

'\x00\x98\x9e\x18Ã\x18?\x98à\x18@    48.02 Û   PhotoÞÛ    CondÞÛ      CiÞÛ  TrmmolÞc  -0.215  -0.000165  -1.64E+03  -0.0026 LeafFan  Flow    Mixer   Temp    Lamp  2  1V     OFF      OFF     OFF   -none- A\n\x01@\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\x01\

## Step 5A — Read the streamed log  *(route A, recommended)*
With the console logging to the COMM port, this captures each record with a timestamp into `li6400_log.csv` and prints it live. Interrupt the cell (■) to stop early.

*Records are delimited per your LogList config (space / tab / comma). This stores the raw line; parse the CSV into columns afterwards to match your setup.*

In [13]:
li.log_to_csv("li6400_log.csv", minutes=5)

logging to li6400_log.csv for 5 min ... (interrupt the cell to stop)
2026-07-17T14:31:48.276 T
2026-07-17T14:31:48.276 @ÛCO2R_æmlÞÛCO2S_æmlÞÛH2OR_mmlÞÛH2OS_mmlÞa   497.7     434.2    16.129    16.910      >> The Pump Is OFF... <<           b   -63.6     0.781      -2.1     48.05 Û   PhotoÞÛ    CondÞÛ      CiÞÛ  TrmmolÞc  -0.218  -0.000168  -1.62E+03  -0.0027 LeafFan  Flow    Mixer   Temp    Lamp  2  1V     OFF      OFF     OFF   -none- A
2026-07-17T14:31:48.277 @                                                                                                                                  R
2026-07-17T14:31:48.277    T
2026-07-17T14:31:48.277 @ÛCO2R_æmlÞÛCO2S_æmlÞÛH2OR_mmlÞÛH2OS_mmlÞa   497.7     434.2    16.125    16.913      >> The Pump Is OFF... <<           b   -63.5     0.788      -2.1

KeyboardInterrupt: 

## Step 5B — On-demand query via LTerm / LPL  *(route B, advanced)*
Requires **LTerm** mode on the console (press **L** at OPEN's main screen). `talk()` sends a line and returns the reply. Start with a bare newline to see the prompt/echo, then try LPL expressions for your OPEN version.

In [ ]:
# LTerm mode must be enabled on the console before this does anything.
# The exact LPL to print a value depends on your OPEN version (manual Ch. 25),
# so treat these as probes to experiment with, not guaranteed commands.

print("--- bare newline (expect a prompt / echo in LTerm) ---")
print(repr(li.talk("", wait=1.0)))

# Example probes to TRY (edit to match your OPEN / LPL):
# print(repr(li.talk("LPLog", wait=2.0)))
# print(repr(li.talk('"CO2R" print', wait=1.5)))

--- bare newline (expect a prompt / echo in LTerm) ---
''


## Step 5C — Read live CO2R / CO2S (parsed from the LTerm mirror)
With the console in **New Measurements mode** and **LTerm On**, this parses the
mirrored display and returns the current reference (**CO2R**) and sample (**CO2S**)
CO2 in µmol/mol, plus H2OR/H2OS in mmol/mol.

Reuses the open `li` from Step 4. If the Step 5A logging cell is running, interrupt
it first (■) — but **don't** close `li`.

In [ ]:
# Parse live CO2R / CO2S (and H2OR / H2OS) from the LTerm display mirror.
# The mirror wraps labels in control bytes (Û...Þ) and sends 'µ' as byte 0xE6,
# so we map every non-printable byte to a space, then read the 4 numbers of the
# 'a' display row that follows the CO2R/CO2S/H2OR/H2OS label block.
import re as _re

_F = r"[-+]?\d+\.?\d*(?:[Ee][-+]?\d+)?"          # int, float, or 1.6E+03
_CO2_ROW = _re.compile(
    rf"CO2R.*?CO2S.*?H2OR.*?H2OS.*?\ba\s+({_F})\s+({_F})\s+({_F})\s+({_F})")

def read_co2(li, seconds=4.0):
    """Latest {'CO2R','CO2S','H2OR','H2OS'} from the LTerm mirror, or None.

    Needs the console in New Measurements mode (values live) with LTerm On.
    CO2 in umol/mol, H2O in mmol/mol.
    """
    raw = li.read_available(seconds)
    text = "".join(chr(b) if 32 <= b <= 126 else " " for b in raw)
    rows = _CO2_ROW.findall(text)
    if not rows:
        print("No CO2 frame parsed. Is the console in New Measurements mode (LTerm On)?")
        return None
    r = rows[-1]                                     # most recent frame
    out = {"CO2R": float(r[0]), "CO2S": float(r[1]),
           "H2OR": float(r[2]), "H2OS": float(r[3])}
    print(f"CO2R (reference) = {out['CO2R']:8.1f} umol/mol")
    print(f"CO2S (sample)    = {out['CO2S']:8.1f} umol/mol")
    print(f"H2OR / H2OS      = {out['H2OR']:.3f} / {out['H2OS']:.3f} mmol/mol")
    print(f"({len(rows)} frames seen in {seconds:g}s)")
    return out

read_co2(li)

## Step 5D — Parse ALL displayed variables
Beyond CO2, grab every labelled row currently on the console screen (Photo, Cond, Ci, Trmmol, H2O, …). `parse_display(li)` returns a `{name: value}` dict of whatever New Measurements is showing.

Put more variables in reach by cycling the display lines on the console — each line you show becomes another key here. (A row hidden behind the *"Pump Is OFF"* banner reappears once the pump is on.) For the *complete* variable set with fixed column names, the logged-record route (Step 5A) is the authoritative source.

In [21]:
# Step 5D (fixed) — parse ALL labelled variables on the console screen.
# The LTerm mirror re-sends the whole screen ~1x/sec, so a capture catches PARTIAL
# frames at its edges. We accept a row ONLY when its label count == value count
# (partial/misaligned frames are dropped — that was the bug that put a CO2 value on
# the H2OR label and lost CO2R), and the most recent COMPLETE copy of each row wins.
import re as _re2

_NUM = r"[-+]?\d+\.?\d*(?:[Ee][-+]?\d+)?"
_GROUP = _re2.compile(r"((?:\xdb[^\xde]*\xde)+)\s*[a-z]\s+(" + _NUM + r"(?:\s+" + _NUM + r")*)")
_LABEL = _re2.compile(r"\xdb\s*([^\xde]*?)\s*\xde")


def _clean_label(lbl):
    ascii_only = "".join(ch if 32 <= ord(ch) <= 126 else "" for ch in lbl).strip()
    return ascii_only.split("_")[0]          # variable name, drop the _unit suffix


def parse_display(li, seconds=6.0, show=True):
    """All labelled variables from the LTerm mirror as {name: float}, partial frames dropped."""
    raw = li.read_available(seconds)
    s = raw.decode("latin-1")
    out, complete = {}, 0
    for grp, nums in _GROUP.findall(s):
        labels = [_clean_label(x) for x in _LABEL.findall(grp)]
        vals = _re2.findall(_NUM, nums)
        if len(labels) != len(vals) or not all(labels):
            continue                          # partial / misaligned frame -> skip
        out.update({lab: float(v) for lab, v in zip(labels, vals)})
        complete += 1
    if show:
        if not out:
            print("No complete frame parsed — capture longer, or check New Measurements / LTerm.")
        else:
            print(f"{len(out)} variables ({complete} complete rows in {seconds:g}s):")
            for k, v in out.items():
                print(f"  {k:8} = {v}")
    return out


parse_display(li)


4 variables (16 complete rows in 6s):
  Photo    = 0.252
  Cond     = -9.25e-05
  Ci       = 4840.0
  Trmmol   = -0.0016


{'Photo': 0.252, 'Cond': -9.25e-05, 'Ci': 4840.0, 'Trmmol': -0.0016}

## Step 5E — Control the flow pump (LPL actuation)
`set_flow(li, umol_per_s)` turns the pump on and sets the chamber flow via the LPL
`FlowSetNewTarget` command (needs **execute-as-LPL** on and the console in **New
Measurements** mode). For **max**, request ~1000 µmol/s and read the achieved rate
from `Flow`/`Flow_ms` with `parse_display(li)` — the pump tops out in the ~700s µmol/s
with a CO2 mixer installed, higher without.

⚠️ This **actuates the instrument**. Running at max increases battery/motor draw and
shrinks the CO2/H2O differentials (fine for a purge, less accurate for assimilation);
keep the chamber mixing fan on. The actuation lines are commented out — uncomment to run.

In [ ]:
# Control the LI-6400 flow PUMP over LTerm (execute-as-LPL must be ON, and the
# console must be in New Measurements mode). LPL is stack-based: value THEN typeCode.
#   FlowSetNewTarget typeCodes:  1 = Pump OFF   2 = Constant flow (umol/s), pump ON
# Ref: OPEN v6 manual Table 25-1 / p.25-19 (FlowSetNewTarget); p.4-3 (max-flow test).

def set_flow(li, umol_per_s, on=True):
    """Set chamber flow (umol/s) and turn the pump on (on=False -> pump off).

    Sends an LPL FlowSetNewTarget command. Requires the console in New Measurements
    mode with 'Incoming Comm: Execute as LPL'. For MAX flow, request a value at or
    above the pump ceiling (e.g. 1000) then read the ACHIEVED rate from Flow /
    Flow_ms via parse_display(li) — the pump tops out ~700s umol/s with a CO2 mixer,
    higher without. Caveat: if a variable/system target was configured for flow,
    OPEN re-applies it every ~30 s and may override this value.
    """
    typecode = 2 if on else 1
    cmd = f"{int(umol_per_s)} {typecode} FlowSetNewTarget"
    li.send(cmd)                      # li.send() appends '\n' (decimal 10 -> LPL executes)
    print("sent:", cmd)


# --- uncomment a line to ACTUATE (this physically changes the instrument) ---
# set_flow(li, 1000)          # request MAX flow, pump ON
# import time; time.sleep(3)  # let it spin up, then check the achieved rate:
# parse_display(li)           # look at Flow / Flow_ms
# set_flow(li, 0, on=False)   # pump OFF

## Step 6 — Clean up
Close the port so other programs (or a re-run of this notebook) can open it.

In [ ]:
li.close()
print("closed", PORT)

closed COM4


## Troubleshooting — link opens but no data
- **Nothing arrives:** confirm the console is actually streaming (Step 5A) or in LTerm mode (Step 5B).
- **Garbage characters:** wrong baud or framing — re-run Step 3, and match data bits / parity / stop bits to the console's COMM config.
- **Still silent with correct settings:** you may need a **null-modem (crossover) adapter** (both PC and console can be DTE, so TX/RX must be swapped), or the **handshaking** doesn't match — set `XONXOFF` / `RTSCTS`, or set handshaking to *None* on the console.
- **Port won't open (Access denied / in use):** another program (PuTTY, a previous run) still holds it — run Step 6, or close the other program.